In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# DBTITLE 1,Create Catalog , Schema and Volume
spark.sql('CREATE CATALOG IF NOT EXISTS crickapidata')
spark.sql('CREATE SCHEMA IF NOT EXISTS crickapidata.org')
spark.sql('CREATE SCHEMA IF NOT EXISTS crickapidata.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS crickapidata.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS crickapidata.gold')
spark.sql('CREATE VOLUME IF NOT EXISTS crickapidata.org.cricket_api_project')

base_path = '/Volumes/crickapidata/org/cricket_api_project'

In [0]:
#Reviewing the data from API
API_KEY='997218a1-49c5-44c4-8efe-795c491ba90a'
api_url=f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"

response =requests.get(api_url)
response.raise_for_status()

api_data=response.json()
print(api_data.keys())
print(json.dumps(api_data,indent= 2)[:2000])


In [0]:
#Writing the API data to base path
raw_file_path = f'{base_path}/current_matches_raw.json'

with open(raw_file_path, 'w') as file:
    json.dump(api_data, file)

print("RAW API data is saved at:", raw_file_path)

In [0]:
#creating bronze schema and bronze data
bronze_schema=StructType([
StructField("source_api",StringType(), True),
StructField("raw_json",StringType(), True),
StructField("ingestion_time",TimestampType(), True)
])

bronze_data=[{
"source_api":api_url,
"raw_json": json.dumps(api_data),
"ingestion_time":None
}]

In [0]:
#Create bronze dataframe

bronze_df = spark.createDataFrame(bronze_data, bronze_schema).withColumn('ingestion_time', current_timestamp())
display(bronze_df)

In [0]:
# Writing created dataframe to bronze layer

bronze_df.write.format('delta').mode('overwrite').saveAsTable('crickapidata.bronze.cricket_bronze_current_matches')

print('Bronze layer is created successfully')

In [0]:
#after this step push the code and in feature_branch saying in description that bronze layer is created